In [63]:
import requests
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [50]:
url = "http://localhost:11434/api/generate"

prompt = """
You are an assistant specialized in answering questions based on the Qur’an only.
Your task is to explain meanings, themes, stories, concepts, ethics, and reflections found in the Qur’an.

Guidelines:
- Base your answers primarily on Qur’anic verses.
- You may explain verses using classical, widely accepted interpretations, but avoid presenting personal opinions as facts.
- Do NOT give legal (fiqh) rulings, fatwas, or detailed jurisprudence.
- If a question is legal or jurisprudential in nature, politely explain that the Qur’an gives principles, not rulings, and provide relevant verses instead.
- Do not rely on hadith unless explicitly asked; if mentioned, clearly label it as supplementary context.
- Maintain a neutral, respectful, and clear tone, suitable for reflection and understanding.
- If a topic is debated or has multiple Qur’anic interpretations, briefly acknowledge the difference without arguing.

Answer style:
- Direct answer first (what happened).
- Narrative explanation in simple, flowing language.
- Key Qur’anic verses referenced.
- (Optional) A short reflection on the meaning of the event, without turning it into rules.

Question:
'What is the only book that is free from any doubt or disbelief?'
"""

# Request payload
payload = {
    "model": "llama3.1",  # replace with your model
    "prompt": prompt,
    "stream": False,
    "max_tokens": 300
}

# Send request
response = requests.post(url, json=payload)
print(response.json()["response"])


**Direct Answer:**
The Book that is free from any doubt or disbelief is the Book of Allah, the Qur'an.

**Narrative Explanation:**
In the Qur'an, there are numerous references to its own authenticity and inviolability. This theme emphasizes the uniqueness and trustworthiness of the scripture as a source of guidance for humanity. The concept is often linked with the idea that the Qur'an confirms previous revelations while being free from any errors or contradictions.

**Key Verse:**
"And We have certainly sent down in this honorable Book (Qur'an) many verses, considering man. And it is a declaration to him of his Lord." - Al-Isra' 17:106
"The revelation of the Book is from Allah, the Mighty and Wise. Do they not consider the Qur'an?" - Fatir 35:31

**Reflection:**
The clarity and integrity of the Qur'an reflect its divine origin. This theme underscores the importance of referring to the scripture for guidance, as it stands as a testament to God's wisdom and mercy. It encourages believer

In [34]:
df = pd.read_csv("question_answering.csv")
df

,Unnamed: 0,question_en,answer_en,chapter_no,verse_list
0,0,What is the only book that is free from any do...,This is the Book of Allah . The evidence: 'Thi...,2,[1 2]
1,1,Are the fruits of paradise similar to the frui...,"Yes, and the evidence: 'And give good news to ...",2,[25]
2,2,How many deaths and how many lives do humans h...,"And you were dead, and He gave you life, then ...",2,[28]
3,3,How many heavens are there?,He it is Who created for you all that is in th...,2,[29]
4,4,"What did Adam learn from Allah , which was no...","He taught Adam the names of all things, then H...",2,[31]
...,...,...,...,...,...
1186,1218,Who is the wife of Ibrahim who laughed when s...,She is Sarah.,11,[71]
1187,1219,"Indeed, Abraham was forbearing, often turning ...","Abraham, peace be upon him, is patient, dislik...",11,[75]
1188,1220,Why did Lot grieve when the guests arrived at...,He feared for them because they were handsome-...,11,[77]
1189,1221,"What does Lut mean by his statement, 'If only...","If I had power and supporters among you, or if...",11,[80]


In [40]:
df.iloc[0]['answer_en']

"This is the Book of Allah . The evidence: 'This is the Book; there is no doubt in it, a guidance for the righteous.'"

In [57]:
r = requests.post(
    "http://localhost:11434/api/embeddings",
    json={
        "model": "qwen3-embedding:8b",
        "prompt": "Joseph dreamed of eleven stars, the sun, and the moon bowing to him."
    }
)

print(json.dumps(r.json(), indent=2))


{
  "embedding": [
    0.01778758317232132,
    0.02932286076247692,
    0.006470706779509783,
    -0.027861827984452248,
    -0.018827125430107117,
    -0.03849893808364868,
    -0.01839672401547432,
    -0.013706566765904427,
    0.0005578480195254087,
    -0.029429646208882332,
    -0.013041050173342228,
    -0.013082676567137241,
    0.025498166680336,
    -0.04163238778710365,
    -0.007280673366039991,
    -0.014641779474914074,
    -0.009460940025746822,
    -0.018942570313811302,
    0.029844466596841812,
    0.021057365462183952,
    -0.003940347116440535,
    0.020214704796671867,
    0.016956180334091187,
    -0.012135419994592667,
    0.021499354392290115,
    0.004041662439703941,
    -0.024175653234124184,
    0.0034863015171140432,
    -0.06322116404771805,
    0.016737239435315132,
    0.02940843068063259,
    -0.020824149250984192,
    -0.00719091109931469,
    0.039567649364471436,
    0.003116520121693611,
    0.018029743805527687,
    -0.008309664204716682,
    0.00

In [60]:
def embed(text):
    r = requests.post(
        "http://localhost:11434/api/embeddings",
        json={
            "model": "qwen3-embedding:8b",
            "prompt": text
        }
    )
    return np.array(r.json()["embedding"])

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))



In [62]:
def gen_response(question):
    prompt = f"""
    You are an assistant specialized in answering questions based on the Qur’an only.
    Your task is to explain meanings, themes, stories, concepts, ethics, and reflections found in the Qur’an.

    Guidelines:
    - Base your answers primarily on Qur’anic verses.
    - You may explain verses using classical, widely accepted interpretations, but avoid presenting personal opinions as facts.
    - Do NOT give legal (fiqh) rulings, fatwas, or detailed jurisprudence.
    - If a question is legal or jurisprudential in nature, politely explain that the Qur’an gives principles, not rulings, and provide relevant verses instead.
    - Do not rely on hadith unless explicitly asked; if mentioned, clearly label it as supplementary context.
    - Maintain a neutral, respectful, and clear tone, suitable for reflection and understanding.
    - If a topic is debated or has multiple Qur’anic interpretations, briefly acknowledge the difference without arguing.

    Answer style:
    - Direct answer first (what happened).
    - Narrative explanation in simple, flowing language.
    - Key Qur’anic verses referenced.
    - (Optional) A short reflection on the meaning of the event, without turning it into rules.

    Question:
    {question}
    """

    # Request payload
    payload = {
        "model": "llama3.1",  # replace with your model
        "prompt": prompt,
        "stream": False,
        "max_tokens": 300
    }

    # Send request
    response = requests.post(url, json=payload)
    return response.json()["response"]


In [61]:
pred = response.json()["response"]
ref  = df.iloc[0]['answer_en']

score = cosine(embed(pred), embed(ref))
print(score)

0.6734629543203067


In [66]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [69]:
scores = []
predictions = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Evaluating"
):
    pred = gen_response(row["question_en"])
    predictions.append(pred)




Evaluating: 100%|██████████| 239/239 [29:09<00:00,  7.32s/it]


In [70]:
for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Evaluating"
):
    score = cosine(embed(pred), embed(row["answer_en"]))
    scores.append(score)

Evaluating: 100%|██████████| 239/239 [02:59<00:00,  1.33it/s]


In [81]:
mean_score_cos_no_train = np.mean(scores)
print(mean_score_cos_no_train)

0.3660297501687958


In [104]:
def judge_llm(question, pred, ref):
    prompt = f"""
    You are an impartial evaluator.

    Given:
    - A question
    - A model-generated answer
    - A reference (ground truth) answer

    Evaluate how well the model answer matches the reference in meaning and correctness.

    Score on a scale from 0 to 5:
    0 = completely incorrect or irrelevant
    1 = mostly incorrect
    2 = partially correct but missing key points
    3 = mostly correct with minor issues
    4 = correct and complete
    5 = perfectly correct and equivalent



    Question:
    {question}

    Model Answer:
    {pred}

    Reference Answer:
    {ref}

    Score from 0 to 5.

    Only output JSON:
    {{"score": number}}
    """

    r = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.1",   # or smaller judge model
            "prompt": prompt,
            "stream": False,
            "temprature": 0
        }
    )

    text = r.json()["response"].strip()
    # safe parse
    import json
    try:
        return json.loads(text)["score"]
    except Exception:
        return None


In [105]:
judge_scores = []
i = 0
for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="LLM judging"
):
    question = row["question_en"]
    ref = row["answer_en"]
    pred = predictions[i]
    score = judge_llm(question, pred, ref)

    judge_scores.append(score)
    i+=1



LLM judging: 100%|██████████| 239/239 [09:45<00:00,  2.45s/it]


In [112]:
total = 0
for i in range(len(judge_scores)):
    if judge_scores[i] != None:
        total += judge_scores[i]
    else:
        print("No judge scores found")

precision_judge = total / (5*(len(judge_scores)-3))
print(precision_judge)

No judge scores found
No judge scores found
No judge scores found
0.5855932203389831
